# 01 — Exchangeable setting: baselines and deep local-covariance score
**Umma Hafsah Himu** — Covariate-Shift-Robust Conformal Prediction Regions for Multivariate Responses with a Deep Local-Covariance Score

Experiment without covariate shift (Parkinson's data). This notebook produces:
a deep mean-and-covariance model with a local-Mahalanobis conformal region on the
Parkinson's telemonitoring data, compared with Gan & Liu (2025) baselines
(marginal coverage and volume, no covariate shift).

Methods:
* **EU** — Gan & Liu Eq. (4): Euclidean-likelihood score with Jackknife calibration, Z = (Y, X).
* **EU-LR** — Gan & Liu Algorithm 1: Z = (Y, m(X)) with linear-regression m, bootstrap calibration (B = 1000).
* **SCP-L2-LR / SCP-M-LR** — split conformal with l2-norm / max-norm of LR residuals (Gan & Liu's comparison methods).
* **Deep-LocalCov** — Braun et al. (2026): network outputs mean f(X) and Cholesky factor L(X), trained by Gaussian NLL;
  score = local Mahalanobis distance; split-conformal calibration.
* **Deep-GlobalCov** (ablation) — same network mean, but one global residual covariance.

Every setting in the config cell is tagged **[paper]** (taken from Gan & Liu or Braun et al.) or **[my choice]** (not specified in the papers).

In [ ]:
import os, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## Configuration

In [ ]:
QUICK = False   # set True only for a fast smoke test

CFG = dict(
    SEED=2026,               # [my choice]
    ALPHA=0.10,              # [paper] Gan & Liu use 90% nominal coverage
    N_TRAIN=1000,            # [paper] Gan & Liu Sec. 5: 1000 training, remaining 4875 test
    N_REPS=20,               # [my choice] Gan & Liu do not state how many random splits Table 1 uses
    B=1000,                  # [paper] Gan & Liu Sec. 4: B = 1000 resamples
    SCP_FIT_FRAC=0.5,        # [my choice] split for SCP baselines (not stated in Gan & Liu)
    DEEP_CAL_FRAC=0.5,       # [my choice] of the 1000 training points: 500 fit / 500 calibration
    DEEP_VAL_FRAC=0.2,       # [my choice] of the fit part, 20% used for early stopping
    HIDDEN=64,               # [my choice]
    N_LAYERS=2,              # [my choice]
    LR=1e-3,                 # [my choice]
    WEIGHT_DECAY=1e-4,       # [my choice]
    BATCH=64,                # [my choice]
    MAX_EPOCHS=500,          # [my choice]
    PATIENCE=30,             # [my choice]
    EPS=1e-3,                # [paper: "small eps" shift on softplus; value is my choice]
)
if QUICK:
    CFG.update(N_REPS=1, B=100, MAX_EPOCHS=40)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = "../results/exchangeable"   # results folder of the repository
os.makedirs(OUT_DIR, exist_ok=True)
CFG

## Data
Download **parkinsons_updrs.data** from the UCI repository (Parkinsons Telemonitoring, dataset 189) and put it in
the repository's `data/` folder. If the file is missing, the cell tries the `ucimlrepo` package instead.

Response Y = (motor_UPDRS, total_UPDRS). Features = the 16 voice measures.

In [ ]:
FEATURES = ['Jitter(%)', 'Jitter(Abs)', 'Jitter:RAP', 'Jitter:PPQ5', 'Jitter:DDP',
            'Shimmer', 'Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'Shimmer:APQ11', 'Shimmer:DDA',
            'NHR', 'HNR', 'RPDE', 'DFA', 'PPE']
TARGETS = ['motor_UPDRS', 'total_UPDRS']
DATA_FILE = "../data/parkinsons_updrs.data"

def load_parkinsons():
    if os.path.exists(DATA_FILE):
        df = pd.read_csv(DATA_FILE)
    else:
        try:
            from ucimlrepo import fetch_ucirepo
            df = fetch_ucirepo(id=189).data.original
        except Exception as e:
            raise FileNotFoundError(
                f"'{DATA_FILE}' not found and ucimlrepo failed ({e}). "
                "Download it from UCI and place it next to this notebook.")
    missing = [c for c in FEATURES + TARGETS if c not in df.columns]
    if missing:
        print("Columns found:", df.columns.tolist())
        raise KeyError(f"Missing columns: {missing}")
    return df, df[FEATURES].to_numpy(float), df[TARGETS].to_numpy(float)

df, X_all, Y_all = load_parkinsons()
print("X:", X_all.shape, "Y:", Y_all.shape)   # expected (5875, 16) and (5875, 2)
df[TARGETS].describe()

## Shared helpers

In [ ]:
def split_quantile(scores, alpha):
    """Split-conformal threshold: ceil((n+1)(1-alpha))-th smallest score (Braun et al. Eq. 2)."""
    n = len(scores)
    k = min(int(math.ceil((n + 1) * (1 - alpha))), n)
    return np.sort(scores)[k - 1]

def unit_ball_volume(q):
    return math.pi ** (q / 2) / math.gamma(q / 2 + 1)

def ellipsoid_volume(t, S):
    """Volume of {y : (y-c)^T S^{-1} (y-c) <= t}."""
    q = S.shape[0]
    return unit_ball_volume(q) * t ** (q / 2) * math.sqrt(np.linalg.det(S))

def ols_fit(X, Y):
    A = np.hstack([np.ones((len(X), 1)), X])
    return np.linalg.lstsq(A, Y, rcond=None)[0]

def ols_predict(beta, X):
    return np.hstack([np.ones((len(X), 1)), X]) @ beta

## Gan & Liu (2025): Euclidean-likelihood (EU) score
EUR(y, x; D) = (y − Ŷ)ᵀ S⁻¹_{yy·x} (y − Ŷ), with Ŷ = Ȳ + S_yx S⁻¹_xx (x − X̄) and S_{yy·x} = S_yy − S_yx S⁻¹_xx S_xy.

In [ ]:
def eur_fit(Y, M):
    q = Y.shape[1]
    S = np.cov(np.hstack([Y, M]), rowvar=False)
    Syy, Sym, Smm = S[:q, :q], S[:q, q:], S[q:, q:]
    Bm = np.linalg.solve(Smm, Sym.T).T            # S_ym S_mm^{-1}
    Syyx = Syy - Bm @ Sym.T
    return dict(Ybar=Y.mean(0), Mbar=M.mean(0), B=Bm, Syyx=Syyx, Sinv=np.linalg.inv(Syyx))

def eur_center(f, M):
    return f['Ybar'] + (np.atleast_2d(M) - f['Mbar']) @ f['B'].T

def eur_score(f, Y, M):
    R = np.atleast_2d(Y) - eur_center(f, M)
    return np.einsum('ij,jk,ik->i', R, f['Sinv'], R)

def run_eu_basic(Xtr, Ytr, Xte, Yte, alpha):
    """Gan & Liu Eq. (4): Jackknife (leave-one-out) calibration with Z = (Y, X)."""
    n = len(Ytr)
    T = np.empty(n)
    keep = np.ones(n, bool)
    for i in range(n):
        keep[i] = False
        T[i] = eur_score(eur_fit(Ytr[keep], Xtr[keep]), Ytr[i], Xtr[i])[0]
        keep[i] = True
    t = np.sort(T)[int(math.ceil(n * (1 - alpha))) - 1]
    f = eur_fit(Ytr, Xtr)
    s = eur_score(f, Yte, Xte)
    return dict(coverage=np.mean(s <= t), covered=(s <= t), volume=ellipsoid_volume(t, f['Syyx']), t=t, f=f)

def run_eu_lr(Xtr, Ytr, Xte, Yte, alpha, B, rng):
    """Gan & Liu Algorithm 1 with m = linear regression and bootstrap calibration."""
    n = len(Ytr)
    beta = ols_fit(Xtr, Ytr)
    f = eur_fit(Ytr, ols_predict(beta, Xtr))
    T = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, n)          # D^b sampled with replacement from D
        j = rng.integers(0, n)               # (Y^b_{n+1}, X^b_{n+1}) sampled from D
        bb = ols_fit(Xtr[idx], Ytr[idx])
        fb = eur_fit(Ytr[idx], ols_predict(bb, Xtr[idx]))
        T[b] = eur_score(fb, Ytr[j], ols_predict(bb, Xtr[j:j + 1]))[0]
    t = np.sort(T)[int(math.ceil(B * (1 - alpha))) - 1]
    s = eur_score(f, Yte, ols_predict(beta, Xte))
    return dict(coverage=np.mean(s <= t), covered=(s <= t), volume=ellipsoid_volume(t, f['Syyx']), t=t)

def run_scp_lr(Xtr, Ytr, Xte, Yte, alpha, rng, fit_frac):
    """Split conformal with l2-norm (SCP-L2-LR) and max-norm (SCP-M-LR) of LR residuals."""
    n, q = Ytr.shape
    perm = rng.permutation(n)
    n_fit = int(n * fit_frac)
    fi, ci = perm[:n_fit], perm[n_fit:]
    beta = ols_fit(Xtr[fi], Ytr[fi])
    Rc = Ytr[ci] - ols_predict(beta, Xtr[ci])
    Rt = Yte - ols_predict(beta, Xte)
    q2 = split_quantile(np.linalg.norm(Rc, axis=1), alpha)
    qM = split_quantile(np.abs(Rc).max(axis=1), alpha)
    c2 = np.linalg.norm(Rt, axis=1) <= q2
    cM = np.abs(Rt).max(axis=1) <= qM
    l2 = dict(coverage=c2.mean(), covered=c2, volume=unit_ball_volume(q) * q2 ** q)
    mx = dict(coverage=cM.mean(), covered=cM, volume=(2 * qM) ** q)
    return l2, mx

## Braun et al. (2026): deep mean + local covariance
p̂(y|x) = N(y | f_θ(x), Σ_φ(x)), Σ_φ(x) = L_φ(x) L_φ(x)ᵀ (Cholesky; diagonal through softplus + ε).
Trained by Gaussian negative log-likelihood (Braun et al. Eq. 5–6).
Covariance head initialised so off-diagonals start near zero and diagonals equal the marginal std of the
(standardised) targets, as described in Braun et al. Appendix A.
Score: s(x, y) = (y − f(x))ᵀ Σ(x)⁻¹ (y − f(x)).

In [ ]:
def softplus_inv(y):
    return math.log(math.expm1(y))

class MeanCovNet(nn.Module):
    def __init__(self, d, k, hidden, n_layers, eps):
        super().__init__()
        layers, din = [], d
        for _ in range(n_layers):
            layers += [nn.Linear(din, hidden), nn.ReLU()]
            din = hidden
        self.trunk = nn.Sequential(*layers)
        self.mean_head = nn.Linear(hidden, k)
        self.chol_head = nn.Linear(hidden, k * (k + 1) // 2)
        self.k, self.eps = k, eps
        rows, cols = torch.tril_indices(k, k)
        self.register_buffer("rows", rows)
        self.register_buffer("cols", cols)
        with torch.no_grad():   # targets are standardised, so marginal std = 1
            self.chol_head.weight.normal_(0.0, 1e-3)
            bias = torch.zeros(k * (k + 1) // 2)
            bias[rows == cols] = softplus_inv(1.0 - eps)
            self.chol_head.bias.copy_(bias)

    def forward(self, x):
        h = self.trunk(x)
        mu = self.mean_head(h)
        raw = self.chol_head(h)
        L = x.new_zeros(x.shape[0], self.k, self.k)
        L[:, self.rows, self.cols] = raw
        diag = F.softplus(torch.diagonal(L, dim1=-2, dim2=-1)) + self.eps
        L = torch.tril(L, diagonal=-1) + torch.diag_embed(diag)
        return mu, L

def whitened(mu, L, y):
    return torch.linalg.solve_triangular(L, (y - mu).unsqueeze(-1), upper=False).squeeze(-1)

def gaussian_nll(mu, L, y):
    z = whitened(mu, L, y)
    logdet_half = torch.log(torch.diagonal(L, dim1=-2, dim2=-1)).sum(-1)
    return (logdet_half + 0.5 * (z ** 2).sum(-1)).mean()

def train_meancov(Xtr, Ytr, Xva, Yva, cfg, seed):
    torch.manual_seed(seed)
    model = MeanCovNet(Xtr.shape[1], Ytr.shape[1], cfg['HIDDEN'], cfg['N_LAYERS'], cfg['EPS']).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg['LR'], weight_decay=cfg['WEIGHT_DECAY'])
    Xt, Yt = [torch.tensor(a, dtype=torch.float32, device=DEVICE) for a in (Xtr, Ytr)]
    Xv, Yv = [torch.tensor(a, dtype=torch.float32, device=DEVICE) for a in (Xva, Yva)]
    best, best_state, wait, hist = float('inf'), None, 0, []
    for ep in range(cfg['MAX_EPOCHS']):
        model.train()
        perm = torch.randperm(len(Xt), device=DEVICE)
        for i in range(0, len(Xt), cfg['BATCH']):
            b = perm[i:i + cfg['BATCH']]
            loss = gaussian_nll(*model(Xt[b]), Yt[b])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            tr = gaussian_nll(*model(Xt), Yt).item()
            va = gaussian_nll(*model(Xv), Yv).item()
        hist.append((tr, va))
        if va < best - 1e-5:
            best, wait = va, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= cfg['PATIENCE']:
                break
    model.load_state_dict(best_state)
    return model, np.array(hist)

def predict(model, X):
    model.eval()
    with torch.no_grad():
        mu, L = model(torch.tensor(X, dtype=torch.float32, device=DEVICE))
    return mu.cpu().numpy(), L.cpu().numpy()

def mahal_np(mu, L, Y):
    z = np.linalg.solve(L, (Y - mu)[..., None])[..., 0]
    return (z ** 2).sum(-1)

def run_deep(Xtr, Ytr, Xte, Yte, alpha, cfg, rng, seed):
    n, k = Ytr.shape
    perm = rng.permutation(n)
    n_fit = int(n * (1 - cfg['DEEP_CAL_FRAC']))
    fit_i, cal_i = perm[:n_fit], perm[n_fit:]
    n_va = int(n_fit * cfg['DEEP_VAL_FRAC'])
    va_i, tr_i = fit_i[:n_va], fit_i[n_va:]

    # standardise with the fit-part statistics only (calibration data stay untouched)
    xm, xs = Xtr[fit_i].mean(0), Xtr[fit_i].std(0) + 1e-12
    ym, ys = Ytr[fit_i].mean(0), Ytr[fit_i].std(0)
    sx = lambda A: (A - xm) / xs
    sy = lambda A: (A - ym) / ys

    model, hist = train_meancov(sx(Xtr[tr_i]), sy(Ytr[tr_i]), sx(Xtr[va_i]), sy(Ytr[va_i]), cfg, seed)

    # --- local covariance (Braun et al.) ---
    mu_c, L_c = predict(model, sx(Xtr[cal_i]))
    qhat = split_quantile(mahal_np(mu_c, L_c, sy(Ytr[cal_i])), alpha)
    mu_t, L_t = predict(model, sx(Xte))
    s_t = mahal_np(mu_t, L_t, sy(Yte))
    sqrt_det = np.prod(np.diagonal(L_t, axis1=1, axis2=2), axis=1) * np.prod(ys)   # back to UPDRS units
    vol_i = unit_ball_volume(k) * qhat ** (k / 2) * sqrt_det
    local = dict(coverage=np.mean(s_t <= qhat), covered=(s_t <= qhat), volume=vol_i.mean(), volume_median=np.median(vol_i),
                 qhat=qhat, vol_i=vol_i, mu_t=mu_t * ys + ym,
                 Sigma_t=np.einsum('i,nij,j->nij', ys, L_t @ np.transpose(L_t, (0, 2, 1)), ys),
                 hist=hist, epochs=len(hist))

    # --- ablation: same mean network, one global covariance ---
    mu_tr, _ = predict(model, sx(Xtr[tr_i]))
    Sg = np.cov(sy(Ytr[tr_i]) - mu_tr, rowvar=False)
    Sg_inv = np.linalg.inv(Sg)
    Rc = sy(Ytr[cal_i]) - mu_c
    qg = split_quantile(np.einsum('ij,jk,ik->i', Rc, Sg_inv, Rc), alpha)
    Rt = sy(Yte) - mu_t
    cg = np.einsum('ij,jk,ik->i', Rt, Sg_inv, Rt) <= qg
    glob = dict(coverage=cg.mean(), covered=cg,
                volume=ellipsoid_volume(qg, Sg) * np.prod(ys))
    return local, glob

## Conditional-coverage metrics (definitions from Klein et al. 2026, Appendix A.2)
* **WSC** (worst-slab coverage, Eq. 10–11): for a direction v, the lowest coverage over all slabs
  {a ≤ vᵀx ≤ b} that contain at least a fraction δ of the test points; then the minimum over random
  directions v on the unit sphere. Closer to 1 − α = 0.90 is better.
* **CEC-X** (coverage error conditional on X, Eq. 12): partition the inputs into J clusters with k-means++,
  compute coverage inside each cluster, and average the squared deviation from 1 − α. Lower is better.

Settings that the paper does not fix are marked **[my choice]** in the config cell below.

In [ ]:
CC = dict(
    WSC_DELTA=0.10,     # [my choice] minimal slab mass delta (Klein et al. leave it free)
    WSC_N_DIR=1000,     # [my choice] number of random directions v
    WSC_GRID=50,        # [my choice] quantile grid for slab end points (exact search is O(n^2) per direction)
    CEC_J=10,           # [my choice] number of k-means++ clusters
)

def standardize_from(Xref, X):
    return (X - Xref.mean(0)) / (Xref.std(0) + 1e-12)

def wsc(Xs, covered_list, directions, delta, grid):
    """Worst-slab coverage for several methods sharing the same directions. Returns one value per method."""
    n = len(Xs)
    m = int(math.ceil(delta * n))
    cut = np.unique(np.linspace(0, n, grid + 1).round().astype(int))
    starts, ends = np.meshgrid(cut, cut, indexing='ij')
    ok = (ends - starts) >= m
    starts, ends = starts[ok], ends[ok]
    best = np.ones(len(covered_list))
    for v in directions:
        order = np.argsort(Xs @ v)
        for k, cov in enumerate(covered_list):
            P = np.concatenate([[0], np.cumsum(cov[order])])
            vals = (P[ends] - P[starts]) / (ends - starts)
            best[k] = min(best[k], vals.min())
    return best

def kmeanspp(X, J, rng, n_iter=100):
    C = [X[rng.integers(len(X))]]
    for _ in range(1, J):
        d2 = np.min(((X[:, None, :] - np.array(C)[None]) ** 2).sum(-1), axis=1)
        C.append(X[rng.choice(len(X), p=d2 / d2.sum())])
    C = np.array(C)
    for _ in range(n_iter):
        lab = np.argmin(((X[:, None, :] - C[None]) ** 2).sum(-1), axis=1)
        newC = np.array([X[lab == j].mean(0) if np.any(lab == j) else C[j] for j in range(J)])
        if np.allclose(newC, C):
            break
        C = newC
    return C

def assign(X, C):
    return np.argmin(((X[:, None, :] - C[None]) ** 2).sum(-1), axis=1)

def cec_x(labels, covered, J, alpha):
    """My reading of Klein et al. Eq. (12): average over test points of the squared deviation of the
    coverage in that point's cluster from 1 - alpha (= cluster-size-weighted mean squared deviation)."""
    cov_j = np.array([covered[labels == j].mean() if np.any(labels == j) else np.nan for j in range(J)])
    return np.nanmean((cov_j[labels] - (1 - alpha)) ** 2), cov_j

## Run all methods over repeated random splits

In [ ]:
rows, keep_rep0, cluster_cov_rep0 = [], {}, {}
t0 = time.time()
N = len(Y_all)
for r in range(CFG['N_REPS']):
    rng = np.random.default_rng(CFG['SEED'] + r)
    perm = rng.permutation(N)
    tr, te = perm[:CFG['N_TRAIN']], perm[CFG['N_TRAIN']:]
    Xtr, Ytr, Xte, Yte = X_all[tr], Y_all[tr], X_all[te], Y_all[te]
    a = CFG['ALPHA']

    eu = run_eu_basic(Xtr, Ytr, Xte, Yte, a)
    eulr = run_eu_lr(Xtr, Ytr, Xte, Yte, a, CFG['B'], rng)
    scp2, scpm = run_scp_lr(Xtr, Ytr, Xte, Yte, a, rng, CFG['SCP_FIT_FRAC'])
    deep, deepg = run_deep(Xtr, Ytr, Xte, Yte, a, CFG, rng, CFG['SEED'] + r)

    methods = [('EU', eu), ('EU-LR', eulr), ('SCP-L2-LR', scp2), ('SCP-M-LR', scpm),
               ('Deep-LocalCov (Braun et al.)', deep), ('Deep-GlobalCov (ablation)', deepg)]

    # conditional-coverage metrics: same directions and clusters for every method in this rep
    Xs_te = standardize_from(Xtr, Xte)
    dirs = rng.normal(size=(CC['WSC_N_DIR'], Xte.shape[1]))
    dirs /= np.linalg.norm(dirs, axis=1, keepdims=True)
    wsc_vals = wsc(Xs_te, [res['covered'].astype(float) for _, res in methods], dirs,
                   CC['WSC_DELTA'], CC['WSC_GRID'])
    centers = kmeanspp(standardize_from(Xtr, Xtr), CC['CEC_J'], rng)   # [my choice] clusters fit on training X
    labels = assign(Xs_te, centers)

    for (name, res), w in zip(methods, wsc_vals):
        cec, cov_j = cec_x(labels, res['covered'].astype(float), CC['CEC_J'], a)
        rows.append(dict(rep=r, method=name, coverage=100 * res['coverage'], volume=res['volume'],
                         volume_median=res.get('volume_median', res['volume']),
                         WSC=100 * w, CEC_X=1e4 * cec,                     # CEC-X reported in (%)^2
                         min_cluster_cov=100 * np.nanmin(cov_j)))
        if r == 0:
            cluster_cov_rep0[name] = (cov_j, np.bincount(labels, minlength=CC['CEC_J']))
    if r == 0:
        keep_rep0 = dict(eu=eu, deep=deep, deepg=deepg, Xte=Xte, Yte=Yte)
    print(f"rep {r + 1}/{CFG['N_REPS']} done | deep epochs={deep['epochs']} | "
          f"deep cov={100 * deep['coverage']:.2f}% | elapsed {time.time() - t0:.0f}s")

res_df = pd.DataFrame(rows)
res_df.to_csv(f"{OUT_DIR}/results_all_reps.csv", index=False)

## Table 1: coverage, volume and conditional coverage, no covariate shift, 90% nominal level
WSC and min_cluster_cov in %, closer to 90 is better. CEC_X in (%)², lower is better.

In [ ]:
order = ['EU', 'EU-LR', 'SCP-L2-LR', 'SCP-M-LR', 'Deep-GlobalCov (ablation)', 'Deep-LocalCov (Braun et al.)']
summary = (res_df.groupby('method')
           .agg(coverage_mean=('coverage', 'mean'), coverage_sd=('coverage', 'std'),
                volume_mean=('volume', 'mean'), volume_sd=('volume', 'std'),
                volume_median=('volume_median', 'mean'),
                WSC_mean=('WSC', 'mean'), WSC_sd=('WSC', 'std'),
                CEC_X_mean=('CEC_X', 'mean'), CEC_X_sd=('CEC_X', 'std'),
                min_cluster_cov=('min_cluster_cov', 'mean'))
           .reindex(order).round(2))
summary.to_csv(f"{OUT_DIR}/table1_summary.csv")
print(f"Reps = {CFG['N_REPS']}, train n = {CFG['N_TRAIN']}, test n = {N - CFG['N_TRAIN']}")
print(summary.to_string())

print("\nReported in Gan & Liu (2025) Table 1 (for reference):")
print(pd.DataFrame({'coverage (%)': [89.11, 89.21, 89.09, 89.91], 'volume': [365.49, 367.40, 1263.71, 1081.94]},
                   index=['EU', 'EU-LR', 'SCP-L2-LR', 'SCP-M-LR']).to_string())

## Figure 1: training curve (rep 1)

In [ ]:
h = keep_rep0['deep']['hist']
plt.figure(figsize=(5, 3.5))
plt.plot(h[:, 0], label='train NLL'); plt.plot(h[:, 1], label='validation NLL')
plt.xlabel('epoch'); plt.ylabel('Gaussian NLL (standardised Y)'); plt.legend(); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_training_curve.png", dpi=200); plt.show()

## Figure 2: the deep region adapts its size to X (rep 1)
The EU region has one volume for every test point; the deep local-covariance region has a different volume for each X.

In [ ]:
vi = keep_rep0['deep']['vol_i']
plt.figure(figsize=(5.5, 3.5))
plt.hist(vi, bins=60, color='tab:blue', alpha=0.7, label='Deep-LocalCov (per test point)')
plt.axvline(keep_rep0['eu']['volume'], color='tab:red', lw=2, label='EU (Gan & Liu), constant')
plt.axvline(keep_rep0['deepg']['volume'], color='tab:green', lw=2, ls='--', label='Deep-GlobalCov, constant')
plt.xlabel('region area (UPDRS$^2$)'); plt.ylabel('count'); plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_volume_distribution.png", dpi=200); plt.show()

## Figure 3: example regions for three test points (small / median / large deep volume)

In [ ]:
def add_ellipse(ax, c, S, t, **kw):
    w, V = np.linalg.eigh(S)
    ang = np.degrees(np.arctan2(V[1, 1], V[0, 1]))
    ax.add_patch(Ellipse(c, 2 * np.sqrt(t * w[1]), 2 * np.sqrt(t * w[0]), angle=ang, fill=False, **kw))

eu0, d0 = keep_rep0['eu'], keep_rep0['deep']
picks = [np.argsort(vi)[int(p * (len(vi) - 1))] for p in (0.05, 0.5, 0.95)]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, i, lab in zip(axes, picks, ['small', 'median', 'large']):
    ce = eur_center(eu0['f'], keep_rep0['Xte'][i])[0]
    add_ellipse(ax, ce, eu0['f']['Syyx'], eu0['t'], color='tab:red', lw=2, label='EU (Gan & Liu)')
    add_ellipse(ax, d0['mu_t'][i], d0['Sigma_t'][i], d0['qhat'], color='tab:blue', lw=2, label='Deep-LocalCov')
    y = keep_rep0['Yte'][i]
    ax.plot(*y, 'k*', ms=12, label='true y')
    ax.set_title(f'{lab} deep volume'); ax.set_xlabel('motor UPDRS'); ax.set_ylabel('total UPDRS')
    ax.autoscale_view(); ax.set_aspect('equal', adjustable='datalim')
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_example_regions.png", dpi=200); plt.show()

## Figure 4: coverage inside each k-means cluster of X (rep 1)
Marginal coverage is ≈ 90% for every method; this plot shows whether that 90% is uniform across groups of inputs.

In [ ]:
show = ['EU', 'Deep-GlobalCov (ablation)', 'Deep-LocalCov (Braun et al.)']
cols = ['tab:red', 'tab:green', 'tab:blue']
J = CC['CEC_J']
sizes = cluster_cov_rep0[show[0]][1]
order_j = np.argsort(sizes)[::-1]
w = 0.27
plt.figure(figsize=(7, 4))
for k, (m, c) in enumerate(zip(show, cols)):
    plt.bar(np.arange(J) + (k - 1) * w, 100 * cluster_cov_rep0[m][0][order_j], width=w, color=c, label=m)
plt.axhline(100 * (1 - CFG['ALPHA']), color='k', ls='--', lw=1, label='target 90%')
plt.xticks(np.arange(J), [f"C{j + 1}\n(n={sizes[j]})" for j in order_j], fontsize=7)
plt.ylabel('coverage within cluster (%)'); plt.ylim(50, 100)
plt.legend(fontsize=7, ncol=4, loc='upper center', bbox_to_anchor=(0.5, -0.22), frameon=False)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_cluster_coverage.png", dpi=200); plt.show()

## Save and print everything I need to see
All tables and figures are saved in `results/exchangeable/`.

In [ ]:
print("CONFIG:", CFG)
print("CC:", CC)
print("\nTABLE 1:\n", summary.to_string())
print("\nDeep epochs per rep:", [len(keep_rep0['deep']['hist'])], "(rep 1)")
print("Files saved in", os.path.abspath(OUT_DIR), ":", sorted(os.listdir(OUT_DIR)))